In [0]:
%sql
USE CATALOG ecommerce_dq;
USE SCHEMA data_quality;

In [0]:
%sql
SELECT current_catalog(),current_schema();

current_catalog(),current_schema()
ecommerce_dq,data_quality


# E-Commerce Data Quality Pipeline
## Project Overview

This project implements a data quality validation pipeline using Databricks and Pyspark. 
The goal is to identify common data quality issues in e-commerce order data before the data is used to analytical purposes.

##Data Quality Checks
- Missing values
- Duplicate records
- Invalid prices
- Invalid quantities
- Invalid dates

In [0]:
from pyspark.sql import functions as F

data = [ 
        (1001, "C001", "Laptop", 1200, 1, "2026-01-10"),
        (1002, "C002", "Mouse", 25.50, 2, "2026-01-11"),
        (1003,"C003", "Keyboaard", 75.00, 1, "2026-01-12"),
        (1003, "C003", "Keyboard", 75.00, 1, "2026-01-12"), # Duplicate
        (1004, None, " Monitor ", 300.00, 1, "2026-01-13"), # Missing customer_id
        (1005, "C005", "Headset", -50.0, 1,
         "2026-01-14"), # Negative price
        (1006, "C006", "Webcam", 80.00, 0, "2026-01-15"), #Invalid quantity
         (1007, "C007", "Microphone", 150.00 ,2, "invalid"), #Invalid date
        (1008, "C008", "Chair", 250.00 , 1, "2026-01-16"),
        (1000, "C009", "Desk", None, 1, "2026-01-18"), #Missing price
        
]

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType,
    IntegerType
)

schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True)
])

df_raw = spark.createDataFrame(data, schema)

display(df_raw)

order_id,customer_id,product,price,quantity,order_date
1001,C001,Laptop,1200.0,1,2026-01-10
1002,C002,Mouse,25.5,2,2026-01-11
1003,C003,Keyboaard,75.0,1,2026-01-12
1003,C003,Keyboard,75.0,1,2026-01-12
1004,null,Monitor,300.0,1,2026-01-13
1005,C005,Headset,-50.0,1,2026-01-14
1006,C006,Webcam,80.0,0,2026-01-15
1007,C007,Microphone,150.0,2,invalid
1008,C008,Chair,250.0,1,2026-01-16
1000,C009,Desk,null,1,2026-01-18


In [0]:
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dq.data_quality.raw_orders")

In [0]:
%sql
SELECT*
FROM ecommerce_dq.data_quality.raw_orders;

order_id,customer_id,product,price,quantity,order_date
1001,C001,Laptop,1200.0,1,2026-01-10
1002,C002,Mouse,25.5,2,2026-01-11
1003,C003,Keyboaard,75.0,1,2026-01-12
1003,C003,Keyboard,75.0,1,2026-01-12
1004,null,Monitor,300.0,1,2026-01-13
1005,C005,Headset,-50.0,1,2026-01-14
1006,C006,Webcam,80.0,0,2026-01-15
1007,C007,Microphone,150.0,2,invalid
1008,C008,Chair,250.0,1,2026-01-16
1000,C009,Desk,null,1,2026-01-18


## 2. Data Profiling

Before applying data quality rules, we inspect the raw dataset to identify potential data quality issues.

The profiling phase focuses on:

- Missing values
- Duplicate records
- Invalid numerical values
- Invalid quantities
- Invalid dates

The objective is to identify data quality issues before the data is used for analytical purposes.

### 2.1 Missing Values

Missing values can affect data analysis and lead to incomplete or inaccurate business insights.

In this check, we count the NULL values in each column of the raw orders dataset.

*Quality rule:* Critical fields should not contain NULL values.

In [0]:
from pyspark.sql import functions as psf

null_counts = df_raw.select([
    psf.count(
        psf.when(psf.col(c).isNull(), c)
    ).alias(c)
    for c in df_raw.columns
])

display(null_counts)

order_id,customer_id,product,price,quantity,order_date
0,1,0,1,0,0


### 2.2 Duplicate Records

Duplicate records can lead to double counting and incorrect business metrics.

In this check, we identify records with duplicated order_id values.

*Quality rule:* Each order should have a unique order_id.

In [0]:
duplicate_orders = (
    df_raw
    .groupBy("order_id")
    .count()
    .filter("count > 1")
)

display(duplicate_orders)

order_id,count
1003,2


### 2.3 Invalid Prices

A product price should be greater than zero.

In this check, we identify records with negative or zero prices.

*Quality rule:* price must be greater than 0.

In [0]:
invalid_prices = (
    df_raw
    .filter((psf.col("price") <= 0) | psf.col("price").isNull())
)

display(invalid_prices)

order_id,customer_id,product,price,quantity,order_date
1005,C005,Headset,-50.0,1,2026-01-14
1000,C009,Desk,null,1,2026-01-18


### 2.4 Invalid Quantities

Order quantities must be greater than zero.

In this check, we identify orders with zero or negative quantities.

*Quality rule:* quantity must be greater than 0.

In [0]:
invalid_quantities = (
    df_raw
    .filter(psf.col("quantity") <= 0)
)

display(invalid_quantities)

order_id,customer_id,product,price,quantity,order_date
1006,C006,Webcam,80.0,0,2026-01-15


### 2.5 Invalid Dates

Order dates must contain valid date values.

In this check, we attempt to convert the order_date column into a date format and identify records that cannot be successfully parsed.

*Quality rule:* order_date must be a valid date.

In [0]:
df_with_dates = df_raw.withColumn(
    "parsed_date",
    psf.expr("try_to_date(order_date, 'yyyy-MM-dd')")
)

invalid_dates = (
    df_with_dates
    .filter(
        psf.col("parsed_date").isNull() &
        psf.col("order_date").isNotNull()
    )
)

display(invalid_dates)

order_id,customer_id,product,price,quantity,order_date,parsed_date
1007,C007,Microphone,150.0,2,invalid,null


## 3. Data Quality Report

The results from each validation rule are consolidated into a single data quality report.

The report provides:

- The validation rule
- The number of affected records
- The validation status
- An overall data quality score

The quality score is calculated based on the percentage of orders that pass all defined quality rules

In [0]:
invalid_records = (
    df_with_dates
    .filter(
        psf.col("customer_id").isNull()
        | psf.col("price").isNull()
        | (psf.col("price") <= 0)
        | (psf.col("quantity") <= 0)
        | psf.col("parsed_date").isNull()
    )
    .select("order_id")
)

duplicate_ids = (
    df_raw
    .groupBy("order_id")
    .count()
    .filter(psf.col("count") > 1)
    .select("order_id")
)

invalid_order_ids = (
    invalid_records
    .union(duplicate_ids)
    .distinct()
)

total_orders = df_raw.select("order_id").distinct().count()
invalid_orders = invalid_order_ids.count()
valid_orders = total_orders - invalid_orders

quality_score = (valid_orders / total_orders) * 100

print(f"Total orders: {total_orders}")
print(f"Orders with quality issues: {invalid_orders}")
print(f"Valid orders: {valid_orders}")
print(f"Data Quality Score: {quality_score:.2f}%")

Total orders: 9
Orders with quality issues: 6
Valid orders: 3
Data Quality Score: 33.33%


## 4. Clean Data

After identifying data quality issues, we create a cleaned version of the dataset.

The raw data is preserved unchanged to maintain data traceability.

The cleaning process removes records that violate the defined quality rules:

- Missing customer IDs
- Missing or invalid prices
- Invalid quantities
- Invalid dates
- Duplicate orders

Only records that pass all validation rules are retained

In [0]:
clean_orders = (
    df_with_dates
    .filter(
        psf.col("customer_id").isNotNull()
        & psf.col("price").isNotNull()
        & (psf.col("price") > 0)
        & (psf.col("quantity") > 0)
        & psf.col("parsed_date").isNotNull()
    )
    .dropDuplicates(["order_id"])
    .drop("parsed_date")
)

display(clean_orders)

order_id,customer_id,product,price,quantity,order_date
1001,C001,Laptop,1200.0,1,2026-01-10
1002,C002,Mouse,25.5,2,2026-01-11
1003,C003,Keyboaard,75.0,1,2026-01-12
1008,C008,Chair,250.0,1,2026-01-16


## 5. Save Clean Data

The validated dataset is stored as a Delta table in Unity Catalog.

The raw dataset remains unchanged, while the cleaned dataset contains only records that comply with the defined data quality rules.

In [0]:
clean_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dq.data_quality.clean_orders")

## 6. Data Quality Results

The results of each validation rule are consolidated into a single quality report.

This report provides visibility into the number of records affected by each quality issue and the corresponding validation status.

A rule is considered *PASS

In [0]:
quality_results = [
    ("Missing Values", 2),
    ("Duplicate Orders", 1),
    ("Invalid Prices", 2),
    ("Invalid Quantities", 1),
    ("Invalid Dates", 1)
]

quality_df = spark.createDataFrame(
    quality_results,
    ["quality_rule", "failed_records"]
)

quality_df = quality_df.withColumn(
    "status",
    psf.when(psf.col("failed_records") == 0, "PASSED")
        .otherwise("FAILED")
)

display(quality_df)

quality_rule,failed_records,status
Missing Values,2,FAILED
Duplicate Orders,1,FAILED
Invalid Prices,2,FAILED
Invalid Quantities,1,FAILED
Invalid Dates,1,FAILED


## 7. Save Data Quality Results

The data quality results are stored as a Delta table in Unity Catalog.

This allows the validation results to be persisted and queried independently from the raw and cleaned datasets.

In [0]:
quality_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dq.data_quality.quality_results")

In [0]:
%sql
SHOW TABLES IN ecommerce_dq.data_quality;

database,tableName,isTemporary
data_quality,clean_orders,false
data_quality,quality_results,false
data_quality,raw_orders,false


In [0]:
%sql
SELECT *
FROM ecommerce_dq.data_quality.quality_results;



quality_rule,failed_records,status
Missing Values,2,FAILED
Duplicate Orders,1,FAILED
Invalid Prices,2,FAILED
Invalid Quantities,1,FAILED
Invalid Dates,1,FAILED


## 8. Project Summary

This project implements a simple data quality pipeline for e-commerce order data using Databricks and PySpark.

### Pipeline

Raw data → Data Quality Validation → Clean Data + Quality Results

### Data Quality Rules

- Missing values
- Duplicate records
- Invalid prices
- Invalid quantities
- Invalid dates

### Technologies

- Databricks
- PySpark
- Python
- SQL
- Delta Lake
- Unity Catalog

### Output Tables

- raw_orders — Original source data
- clean_orders — Validated records
- quality_results — Data quality validation results

### Business Value

The pipeline identifies data quality issues before the data is used for analytical purposes, helping prevent inaccurate reporting and unreliable business decisions.

In [0]:
%sql
SELECT 
    'raw_orders' AS table_name,
    COUNT(*) AS records
FROM ecommerce_dq.data_quality.raw_orders

UNION ALL

SELECT 
    'clean_orders',
    COUNT(*)
FROM ecommerce_dq.data_quality.clean_orders

UNION ALL

SELECT 
    'quality_results',
    COUNT(*)
FROM ecommerce_dq.data_quality.quality_results;

table_name,records
raw_orders,10
clean_orders,4
quality_results,5
